## 表格数据预处理函数

使用openpyxl将合并单元格拆分, 生成中间文件, 读取中间文件

In [136]:
import openpyxl


# 拆分所有的合并单元格, 并赋予合并之前的值
def unmerge_and_fill_cells(worksheet, columns_to_clean=None):
    # 获取工作表中所有合并单元格的范围
    all_merged_cell_ranges = list(worksheet.merged_cells.ranges)

    # 遍历所有合并单元格的范围
    for merged_cell_range in all_merged_cell_ranges:
        merged_cell = merged_cell_range.start_cell  # 获取合并单元格范围的起始单元格
        worksheet.unmerge_cells(range_string=merged_cell_range.coord)  # 取消合并单元格

        # 遍历合并单元格范围内的每个单元格
        for row_index, col_index in merged_cell_range.cells:
            cell = worksheet.cell(row=row_index, column=col_index)  # 获取当前单元格对象
            cell.value = merged_cell.value  # 将起始单元格的值赋予当前单元格

            # 如果指定了需要清理的列，并且当前单元格在这些列中
            if columns_to_clean and col_index in columns_to_clean:
                cell.value = str(cell.value).replace('\n', '') if cell.value is not None else None  # 去除单元格中的换行符


# 读取原始xlsx文件，拆分并填充单元格，然后生成中间临时文件。
def unmerge_cell(filename, columns_to_clean=None):
    wb = openpyxl.load_workbook(filename)  # 加载原始 Excel 文件
    # 遍历工作簿中的所有工作表
    for sheet_name in wb.sheetnames:
        sheet = wb[sheet_name]  # 获取工作表对象
        unmerge_and_fill_cells(sheet, columns_to_clean)  # 调用 unmerge_and_fill_cells 函数处理当前工作表
    filename = filename.replace(".xlsx", "_temp.xlsx")  # 生成新的文件名，将原始文件名中的 ".xlsx" 替换为 "_temp.xlsx"
    wb.save(filename)  # 保存修改后的工作簿到新的文件
    wb.close()  # 关闭工作簿，释放资源

    return filename  # 返回新的文件名

## 加载附件1

In [137]:
import pandas as pd

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)

In [138]:
Annex_1 = unmerge_cell("附件1.xlsx")

# 附件1 乡村的现有耕地
existing_cultivated_land_Annex_1 = pd.read_excel(Annex_1,
                                                 engine="openpyxl",
                                                 sheet_name="乡村的现有耕地",
                                                 usecols="A:C",
                                                 nrows=55,
                                                 header=0)

# 附件1 乡村种植的农作物
crops_grown_Annex_1 = pd.read_excel(Annex_1,
                                    engine="openpyxl",
                                    sheet_name="乡村种植的农作物",
                                    usecols="A:D",
                                    nrows=42,
                                    header=0)

## 加载附件2

In [141]:
Annex_2 = unmerge_cell("附件2.xlsx")

# 附件2 2023年的农作物种植情况
crop_planting_in_2023_Annex_2 = pd.read_excel(Annex_2,
                                              engine="openpyxl",
                                              sheet_name="2023年的农作物种植情况",
                                              usecols="A:F",
                                              nrows=88,
                                              header=0)

# 附件2 2023年统计的相关数据
relevant_data_for_the_year_2023_Annex_2 = pd.read_excel(Annex_2,
                                                        engine="openpyxl",
                                                        sheet_name="2023年统计的相关数据",
                                                        usecols="A:H",
                                                        nrows=108,
                                                        header=0)

In [142]:
# 打印 2023年的农作物种植情况 表头名
print("2023年的农作物种植情况", crop_planting_in_2023_Annex_2.columns.values)

# 打印 2023年统计的相关数据 表头名
print("2023年统计的相关数据", relevant_data_for_the_year_2023_Annex_2.columns.values)

2023年的农作物种植情况 ['种植地块' '作物编号' '作物名称' '作物类型' '种植面积/亩' '种植季次']
2023年统计的相关数据 ['序号' '作物编号' '作物名称' '地块类型' '种植季次' '亩产量/斤' '种植成本/(元/亩)' '销售单价/(元/斤)']


## 处理数据

### 将统计数据 销售单价/(元/斤) 取均值

In [143]:
column_quarter_data = relevant_data_for_the_year_2023_Annex_2['销售单价/(元/斤)']

# 创建一个新列来存储每个范围的平均值
average_values = []

# 遍历每个值，计算平均值
for value in column_quarter_data:
    # 将字符串拆分为最小值和最大值
    min_val, max_val = map(float, value.split('-'))
    # 计算平均值
    average = (min_val + max_val) / 2
    average_values.append(average)

# 将计算结果添加到DataFrame中
relevant_data_for_the_year_2023_Annex_2['平均销售单价/(元/斤)'] = average_values

relevant_data_for_the_year_2023_Annex_2

,序号,作物编号,作物名称,地块类型,种植季次,亩产量/斤,种植成本/(元/亩),销售单价/(元/斤),平均销售单价/(元/斤)
0,1,1,黄豆,平旱地,单季,400,400,2.50-4.00,3.25
1,2,2,黑豆,平旱地,单季,500,400,6.50-8.50,7.50
2,3,3,红豆,平旱地,单季,400,350,7.50-9.00,8.25
3,4,4,绿豆,平旱地,单季,350,350,6.00-8.00,7.00
4,5,5,爬豆,平旱地,单季,415,350,6.00-7.50,6.75
5,6,6,小麦,平旱地,单季,800,450,3.00-4.00,3.50
6,7,7,玉米,平旱地,单季,1000,500,2.50-3.50,3.00
7,8,8,谷子,平旱地,单季,400,360,6.00-7.50,6.75
8,9,9,高粱,平旱地,单季,630,400,5.50-6.50,6.00
9,10,10,黍子,平旱地,单季,525,360,6.50-8.50,7.50


## 写入数据

### 写入数据预处理

In [144]:
columns_to_clean = [1]

Result_1_1 = unmerge_cell("result1_1.xlsx", columns_to_clean)

pd.read_excel(Result_1_1,
              engine="openpyxl",
              sheet_name="2024",
              usecols="A:AQ",
              nrows=83,
              header=0)

C:\Dev\DevKit\anaconda3\envs\machine-learning\lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,Unnamed: 0,地块名,黄豆,黑豆,红豆,绿豆,爬豆,小麦,玉米,谷子,高粱,黍子,荞麦,南瓜,红薯,莜麦,大麦,水稻,豇豆,刀豆,芸豆,土豆,西红柿,茄子,菠菜,青椒,菜花,包菜,油麦菜,小青菜,黄瓜,生菜,辣椒,空心菜,黄心菜,芹菜,大白菜,白萝卜,红萝卜,榆黄菇,香菇,白灵菇,羊肚菌
0,第一季,A1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,第一季,A2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,第一季,A3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,第一季,A4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,第一季,A5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,第一季,A6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,第一季,B1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,第一季,B2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,第一季,B3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,第一季,B4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
from openpyxl import load_workbook


def write_to_excel(season, plot_name, product, value, file_path, sheet_name):
    book = load_workbook(file_path)
    sheet = book[sheet_name]

    # 找到对应的行
    row = None
    for idx, row_data in enumerate(sheet.iter_rows(min_row=2, values_only=True)):
        if row_data[0] == season and row_data[1] == plot_name:
            row = idx + 2  # 加2是因为第一行是表头
            break
    if row is None:
        return

    # 找到对应的列
    col = None
    for idx, cell in enumerate(sheet[1]):
        if cell.value == product:
            col = idx  # 列索引
            break
    if col is None:
        return

    # 写入数据
    sheet.cell(row=row, column=col + 1, value=value)  # column+1是因为第一列是季度
    book.save(file_path)

    return file_path

In [146]:
# 示例使用
write_to_excel('第一季', 'A1', '黄豆', 100, Result_1_1, '2024')
write_to_excel('第一季', 'A2', '黑豆', 100, Result_1_1, '2024')

pd.read_excel(Result_1_1,
              engine="openpyxl",
              sheet_name="2024",
              usecols="A:AQ",
              nrows=83,
              header=0)

,Unnamed: 0,地块名,黄豆,黑豆,红豆,绿豆,爬豆,小麦,玉米,谷子,高粱,黍子,荞麦,南瓜,红薯,莜麦,大麦,水稻,豇豆,刀豆,芸豆,土豆,西红柿,茄子,菠菜,青椒,菜花,包菜,油麦菜,小青菜,黄瓜,生菜,辣椒,空心菜,黄心菜,芹菜,大白菜,白萝卜,红萝卜,榆黄菇,香菇,白灵菇,羊肚菌
0,第一季,A1,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,第一季,A2,NaN,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,第一季,A3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,第一季,A4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,第一季,A5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,第一季,A6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,第一季,B1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,第一季,B2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,第一季,B3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,第一季,B4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [147]:
def copy_data(source_file_path, source_sheet_name, target_file_path, target_sheet_name, data_range, target_range):
    # 加载源Excel文件
    source_book = load_workbook(source_file_path)
    source_sheet = source_book[source_sheet_name]

    # 加载目标Excel文件
    target_book = load_workbook(target_file_path)
    target_sheet = target_book[target_sheet_name]

    # 解析数据范围
    src_start_cell = source_sheet[data_range.split(':')[0]]
    src_end_cell = source_sheet[data_range.split(':')[1]]
    src_start_row = src_start_cell.row
    src_start_col = src_start_cell.column
    src_end_row = src_end_cell.row
    src_end_col = src_end_cell.column

    # 解析目标起始单元格
    target_start_cell = target_sheet[target_range]
    target_start_row = target_start_cell.row
    target_start_col = target_start_cell.column

    # 复制数据
    for row in range(src_start_row, src_end_row + 1):
        for col in range(src_start_col, src_end_col + 1):
            source_cell = source_sheet.cell(row=row, column=col)
            target_cell = target_sheet.cell(row=target_start_row + (row - src_start_row), column=target_start_col + (col - src_start_col))
            target_cell.value = source_cell.value

    # 保存目标Excel文件
    target_book.save(target_file_path)

In [147]:
# 示例使用
source_file_path = 'result1_1_temp.xlsx'  # 源文件路径
source_sheet_name = '2024'  # 源工作表名称
target_file_path = 'result1_1 - 副本.xlsx'  # 目标文件路径
target_sheet_name = '2024'  # 目标工作表名称
data_range = 'C2:AQ83'  # 要复制的数据范围（例如：A1:AQ83）
target_range = 'C2'  # 目标工作表中要粘贴数据的起始单元格（例如：A1）

copy_data(source_file_path, source_sheet_name, target_file_path, target_sheet_name, data_range, target_range)